# PCA Applications: Compression, Visualisation, and Noise Reduction

## Learning Objectives
- Use PCA for **data compression**: represent high-dimensional data with fewer components
- Apply PCA for **2D/3D visualisation** of high-dimensional datasets
- Use PCA for **noise reduction**: low-rank approximation strips noisy directions
- Understand the **compression ratio** and trade-offs between $k$ and reconstruction quality

## Problem Statement

Given data $x^{(i)} \in \mathbb{R}^n$, PCA finds a $k$-dimensional representation $y^{(i)} \in \mathbb{R}^k$ via:
$$y^{(i)} = U_k^T(x^{(i)} - \mu), \qquad \hat{x}^{(i)} = \mu + U_k y^{(i)}$$

where $U_k = [u_1,\ldots,u_k]$ contains the top-$k$ eigenvectors of $\Sigma = \frac{1}{m}\sum_i x^{(i)}{x^{(i)}}^T$.

**Applications:**

1. **Compression**: Store $y^{(i)} \in \mathbb{R}^k$ and $U_k$ instead of $x^{(i)} \in \mathbb{R}^n$. Compression ratio $\approx n/k$.

2. **Visualisation**: Project to $k=2$ or $k=3$ for human-interpretable scatter plots.

3. **Noise reduction**: If signal lives in a low-dimensional subspace, discarding small eigenvalue directions removes noise:
$$\hat{x}^{(i)} = \mu + \sum_{j=1}^k (u_j^T (x^{(i)}-\mu))\, u_j$$

**Memory saved** by compressing $m$ points from $\mathbb{R}^n$ to $\mathbb{R}^k$:
- Original: $mn$ numbers
- Compressed: $mk + nk$ numbers (projections + basis)
- Ratio: $\frac{mk + nk}{mn} = \frac{k(m+n)}{mn}$

## 1. PCA for Data Compression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X = faces.data   # (400, 4096)
m, n = X.shape
img_shape = (64, 64)

k_values = [1, 5, 10, 20, 40, 80, 150]
errors, ratios = [], []

for k in k_values:
    pca = PCA(n_components=k, random_state=0)
    Y   = pca.fit_transform(X)
    X_r = pca.inverse_transform(Y)
    mse = np.mean((X - X_r) ** 2)
    errors.append(mse)
    # Memory: store Y (m*k) + components (k*n) + mean (n)
    mem_compressed = m * k + k * n + n
    mem_original   = m * n
    ratios.append(mem_compressed / mem_original)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(k_values, errors, 'b-o', lw=2.5, ms=6)
for k, e in zip(k_values, errors):
    ax.annotate(f'k={k}', (k, e), textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Number of components $k$')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction error vs compression level')

ax = axes[1]
ax.plot(k_values, [r * 100 for r in ratios], 'r-o', lw=2.5, ms=6)
ax.axhline(100, color='k', ls='--', lw=1.5, label='No compression (100%)')
for k, r in zip(k_values, ratios):
    ax.annotate(f'{r*100:.1f}%', (k, r*100), textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Number of components $k$')
ax.set_ylabel('Memory used (% of original)')
ax.set_title('Memory cost vs compression level')
ax.legend(fontsize=9)

fig.suptitle(f'PCA Compression: {m} face images ({n}-dim each)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('k     MSE       Memory %   Compression ratio')
for k, e, r in zip(k_values, errors, ratios):
    print(f'{k:4d}  {e:.4f}    {r*100:5.1f}%      {1/r:.1f}x')

## 2. PCA for High-Dimensional Visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
X_d, y_d = digits.data, digits.target  # (1797, 64), labels 0-9

pca2 = PCA(n_components=2, random_state=0)
pca3 = PCA(n_components=3, random_state=0)
Y2 = pca2.fit_transform(X_d)
Y3 = pca3.fit_transform(X_d)

colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig = plt.figure(figsize=(15, 6))

# 2D PCA projection
ax1 = fig.add_subplot(1, 2, 1)
for digit in range(10):
    mask = y_d == digit
    ax1.scatter(Y2[mask, 0], Y2[mask, 1], s=12, alpha=0.6,
                color=colors[digit], label=str(digit))
ax1.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% var)')
ax1.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% var)')
ax1.set_title('2D PCA projection of digits dataset\n(64-dim → 2-dim)')
ax1.legend(title='Digit', ncol=2, fontsize=8, markerscale=2)

# 3D PCA projection
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
for digit in range(10):
    mask = y_d == digit
    ax2.scatter(Y3[mask, 0], Y3[mask, 1], Y3[mask, 2],
                s=8, alpha=0.5, color=colors[digit], label=str(digit))
ax2.set_xlabel(f'PC1')
ax2.set_ylabel(f'PC2')
ax2.set_zlabel(f'PC3')
ax2.set_title(f'3D PCA projection\n({sum(pca3.explained_variance_ratio_)*100:.1f}% var explained)')

fig.suptitle('PCA for Visualisation: 64-dim Digits Dataset Projected to 2D/3D',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Cumulative variance
pca_full = PCA(random_state=0).fit(X_d)
cum_var  = np.cumsum(pca_full.explained_variance_ratio_)
k90 = np.searchsorted(cum_var, 0.90) + 1
k95 = np.searchsorted(cum_var, 0.95) + 1
print(f'Original dimensionality: {X_d.shape[1]}')
print(f'Components for 90% variance: {k90}')
print(f'Components for 95% variance: {k95}')
print(f'Top 2 components explain: {cum_var[1]*100:.1f}%')

## 3. PCA for Noise Reduction

If the signal lies in a low-dimensional subspace and noise is spread across all dimensions,
projecting to the top-$k$ subspace **keeps signal while discarding noise**:

- Signal directions: large eigenvalues (high variance)
- Noise directions: small eigenvalues (low variance)

Reconstruction $\hat{x} = \mu + U_k U_k^T (x - \mu)$ averages out noise in the discarded directions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(7)

# Generate: signal in 2D subspace of 50D space, noisy observations
n_signal, n_total, m_samples = 2, 50, 200
noise_level = 0.8

# True low-rank signal
U_true = np.linalg.qr(np.random.randn(n_total, n_signal))[0][:, :n_signal]
z_true = np.random.randn(m_samples, n_signal) * np.array([3.0, 1.5])
X_clean = z_true @ U_true.T

# Add Gaussian noise
noise   = np.random.randn(m_samples, n_total) * noise_level
X_noisy = X_clean + noise

# Denoise with PCA at various k
k_vals = [1, 2, 5, 10, 20]
snrs   = []

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# First two dimensions of clean vs noisy vs denoised
ax = axes[0]
ax.scatter(X_clean[:, 0], X_clean[:, 1], s=20, alpha=0.6, c='#2166ac', label='Clean signal')
ax.scatter(X_noisy[:, 0], X_noisy[:, 1], s=20, alpha=0.3, c='gray',    label='Noisy')

pca_d = PCA(n_components=n_signal, random_state=0)
X_den = pca_d.inverse_transform(pca_d.fit_transform(X_noisy))
ax.scatter(X_den[:, 0], X_den[:, 1], s=20, alpha=0.6, c='#d6604d',
           label=f'Denoised (k={n_signal})')
ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2')
ax.set_title('First 2 dimensions\nClean → Noisy → Denoised')
ax.legend(fontsize=8.5)

# SNR vs k
for k in k_vals:
    pca_k = PCA(n_components=k, random_state=0)
    X_r   = pca_k.inverse_transform(pca_k.fit_transform(X_noisy))
    signal_power = np.mean(X_clean ** 2)
    residual     = np.mean((X_r - X_clean) ** 2)
    snrs.append(10 * np.log10(signal_power / residual))

ax = axes[1]
ax.plot(k_vals, snrs, 'g-o', lw=2.5, ms=8)
ax.axvline(n_signal, color='r', ls='--', lw=2, label=f'True rank = {n_signal}')
ax.set_xlabel('Number of components $k$')
ax.set_ylabel('SNR (dB)')
ax.set_title('Signal-to-noise ratio vs $k$\n(peak near true signal rank)')
ax.legend(fontsize=9)

# Eigenvalue spectrum: signal vs noise
ax = axes[2]
pca_full = PCA(random_state=0).fit(X_noisy)
eigvals  = pca_full.explained_variance_
ax.semilogy(range(1, len(eigvals)+1), eigvals, 'b-', lw=2)
ax.axvspan(0.5, n_signal+0.5, alpha=0.15, color='green', label='Signal subspace')
ax.axvspan(n_signal+0.5, len(eigvals)+0.5, alpha=0.1, color='red', label='Noise subspace')
ax.set_xlabel('Component index')
ax.set_ylabel('Eigenvalue (log scale)')
ax.set_title('Scree plot: gap between signal and noise')
ax.legend(fontsize=9)

fig.suptitle(f'PCA for Noise Reduction: {n_signal}-dim signal in {n_total}-dim space',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Noise level σ = {noise_level}')
for k, snr in zip(k_vals, snrs):
    print(f'k={k:2d}: SNR = {snr:.2f} dB')

## 4. Image Compression: Faces at Different Compression Levels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X = faces.data
img_shape = (64, 64)

k_show = [1, 5, 10, 25, 50, 100, 200, 400]
n_faces_show = 3

fig, axes = plt.subplots(n_faces_show, len(k_show) + 1, figsize=(16, 6))

for row in range(n_faces_show):
    # Original
    axes[row, 0].imshow(X[row].reshape(img_shape), cmap='gray', vmin=0, vmax=1)
    axes[row, 0].axis('off')
    if row == 0:
        axes[row, 0].set_title('Original\n(4096-dim)', fontsize=8)

    for col, k in enumerate(k_show, start=1):
        pca_k = PCA(n_components=k, random_state=0)
        Y_k   = pca_k.fit_transform(X)
        X_r   = pca_k.inverse_transform(Y_k)
        img   = np.clip(X_r[row].reshape(img_shape), 0, 1)
        axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if row == 0:
            mem_pct = (k * (len(X) + 4096) + 4096) / (len(X) * 4096) * 100
            var_exp = pca_k.explained_variance_ratio_.sum() * 100
            axes[row, col].set_title(f'k={k}\n{var_exp:.0f}% var', fontsize=7)

fig.suptitle('Face Compression at Different PCA Component Counts',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Original: {X.shape[1]} dimensions per face')
print(f'\nk    Var%   PSNR(dB)')
for k in k_show:
    pca_k = PCA(n_components=k, random_state=0)
    X_r   = pca_k.inverse_transform(pca_k.fit_transform(X))
    mse   = np.mean((X - X_r) ** 2)
    psnr  = 10 * np.log10(1.0 / mse) if mse > 0 else float('inf')
    var   = pca_k.explained_variance_ratio_.sum() * 100
    print(f'{k:4d}  {var:5.1f}%  {psnr:.1f}')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">PCA basis</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">U&#x2096; = [u&#x2081;,...,u&#x2096;]</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >learned from training data via covariance eigendecomposition</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >project: z&#x2071; = U&#x2096;&#x1d40; x&#x2071;</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">k-dim code</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">z&#x2071; &#x2208; &#x211d;&#x1d55c;</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >compression: store k floats vs n floats (k/n ratio)</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >reconstruct: x&#x303;&#x2071; = U&#x2096; z&#x2071;</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Applications</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >compression, 2-D/3-D visualisation, noise reduction / denoising</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >retain enough variance (90&#x2013;99%) or choose k by elbow</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Decision:</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">choose k</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >scree plot elbow or fraction-variance-explained threshold</text>
</svg>
""")

## Summary

| Application | How PCA is used | Key Parameter |
|---|---|---|
| Data compression | Store $z^{(i)} \in \mathbb{R}^k$ instead of $x^{(i)} \in \mathbb{R}^n$ | Compression ratio $k/n$ |
| Visualisation | Project to $k=2$ or $k=3$ for scatter plots | Always 2 or 3 |
| Noise reduction | Reconstruct $\tilde{x} = U_k z$ — small eigenvalues absorb noise | Choose $k$ by variance explained |

| Concept | Formula / Rule | Key Insight |
|---|---|---|
| Explained variance | $\sum_{j=1}^k \lambda_j / \sum_j \lambda_j$ | Target 90–99% retained variance |
| Reconstruction MSE | $\sum_{j=k+1}^n \lambda_j$ | Sum of dropped eigenvalues |
| Scree plot | Eigenvalue vs. component index | Look for elbow to choose $k$ |

**Key insight:** PCA enables compression, visualisation, and denoising through a single linear projection; the number of components $k$ is chosen by the fraction of variance retained or a scree-plot elbow — there is no universally optimal $k$.